# Wikipedia airborne-radar knowledge graph with Neo4j

This notebook uses the repository's `combat_id_calibration.graph_ingest` module to build a local Neo4j knowledge graph from Wikipedia pages for the MiG-29, the Ukrainian Air Force, the Russian Air Force, the Bars radar, and representative Russian or Israeli airborne radars.

The existing ingestion module performs the core workflow: fetch Wikipedia HTML, extract readable text, chunk documents, ask a local Ollama model to return auditable JSON facts, optionally write facts to JSONL, and populate Neo4j with `Entity`, `Source`, `FACT`, and `MENTIONED_IN` records.

> Wikipedia is a convenient public source, but it is not authoritative. Review `facts-jsonl` output before using extracted relationships for combat-identification scoring or calibration.


## 1. Install and runtime prerequisites

From the repository root, install the graph extra and make sure Ollama is running with the configured model:

```bash
python -m pip install -e .[graph]
ollama pull qwen3.5:9b
ollama serve
```

If your notebook starts in `notebooks/`, the setup cell below adds the repository root to `sys.path` so it imports the local module under development.


In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from combat_id_calibration.graph_ingest import (
    DEFAULT_MODEL,
    DEFAULT_OLLAMA_URL,
    extract_facts,
    load_documents,
    populate_neo4j,
    write_facts_jsonl,
)


## 2. Configure Wikipedia sources

The first four URLs satisfy the explicitly requested pages. The remaining URLs add representative Russian and Israeli airborne radar pages so the resulting graph has more radar-specific evidence.


In [2]:
NEO4J_URI = 'bolt://localhost:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = None  # replace with your Neo4j password before running the population cell
NEO4J_DATABASE = None
FACTS_JSONL = REPO_ROOT / 'wikipedia_airborne_radars_facts.jsonl'
MODEL = DEFAULT_MODEL
OLLAMA_URL = DEFAULT_OLLAMA_URL
MAX_CHARS = 6000
OVERLAP = 500

WIKIPEDIA_URLS = [
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-29',
    'https://en.wikipedia.org/wiki/Ukrainian_Air_Force',
    'https://en.wikipedia.org/wiki/Russian_Air_Force',
    'https://en.wikipedia.org/wiki/Bars_radar',

    # Additional Russian airborne radars
    'https://en.wikipedia.org/wiki/Irbis-E',
    'https://en.wikipedia.org/wiki/Zhuk_(radar)',
    'https://en.wikipedia.org/wiki/Mech_radar',
    # Additional Israeli airborne radars
    'https://en.wikipedia.org/wiki/EL/M-2032',
    'https://en.wikipedia.org/wiki/EL/W-2085',
    'https://en.wikipedia.org/wiki/EL/M-2052',
]


## 3. Load Wikipedia documents with `graph_ingest.py`

`load_documents` delegates each Wikipedia URL to `read_wikipedia`, preserving a deterministic source ID, source type, locator URL, article title, and extracted article text.


In [3]:
documents = load_documents(pdf_paths=[], wikipedia_urls=WIKIPEDIA_URLS)
print(f'Loaded {len(documents)} Wikipedia documents')
for document in documents:
    print(f'- {document.title}: {len(document.text):,} characters from {document.locator}')


https://en.wikipedia.org/wiki/Mikoyan_MiG-29
https://en.wikipedia.org/wiki/Ukrainian_Air_Force
https://en.wikipedia.org/wiki/Russian_Air_Force
https://en.wikipedia.org/wiki/Bars_radar
https://en.wikipedia.org/wiki/Irbis-E
https://en.wikipedia.org/wiki/Zhuk_(radar)
https://en.wikipedia.org/wiki/Mech_radar
https://en.wikipedia.org/wiki/EL/M-2032
https://en.wikipedia.org/wiki/EL/W-2085
https://en.wikipedia.org/wiki/EL/M-2052
Loaded 10 Wikipedia documents
- Mikoyan MiG-29: 174,975 characters from https://en.wikipedia.org/wiki/Mikoyan_MiG-29
- Ukrainian Air Force: 65,664 characters from https://en.wikipedia.org/wiki/Ukrainian_Air_Force
- Russian Air Force: 61,002 characters from https://en.wikipedia.org/wiki/Russian_Air_Force
- Bars radar: 10,195 characters from https://en.wikipedia.org/wiki/Bars_radar
- Irbis-E: 7,406 characters from https://en.wikipedia.org/wiki/Irbis-E
- Zhuk (radar): 19,065 characters from https://en.wikipedia.org/wiki/Zhuk_(radar)
- Mech radar: 5,336 characters from ht

In [4]:
documents[5]

SourceDocument(source_id='02bea5aaf367bd98', source_type='wikipedia', locator='https://en.wikipedia.org/wiki/Zhuk_(radar)', title='Zhuk (radar)', text='Zhuk (radar) - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation \n Main page \n\n Contents \n\n Current events \n\n Random article \n\n About Wikipedia \n\n Contact us \n Contribute \n Help \n\n Learn to edit \n\n Community portal \n\n Recent changes \n\n Upload file \n\n Special pages \n Search Search Appearance \n Donate \n\n Create account \n\n Log in \n Personal tools \n Donate \n\n Create account \n\n Log in \n\n Contents \n move to sidebar hide \n (Top) \n\n 1 Description \n\n 2 Variants Toggle Variants subsection \n 2.1 Zhuk \n\n 2.2 Zhuk-8II \n\n 2.3 Zhemchoug \n\n 2.4 Zhuk-10PD \n\n 2.5 Zhuk-27 \n\n 2.6 Zhuk-M (Export Designation Zhuk-ME) \n\n 2.7 Zhuk-MS (Export Designation Zhuk-MSE) \n\n 2.8 Zhuk-F/Zhuk-PH \n\n 2.9 RP-35 \n\n 2.10 Zhuk-MF (Export Designation Zhuk-MFE) formerly part of the Sokol PE

## 4. Extract auditable facts with Ollama

This cell calls the repository ingestion module's `extract_facts`, which chunks each source and applies the module's constrained JSON extraction prompt. Keep `FACTS_JSONL` under review; it is the audit artifact to inspect before trusting the Neo4j graph.


In [5]:
facts = extract_facts(
    documents[5:6],
    model=MODEL,
    ollama_url=OLLAMA_URL,
    max_chars=500,
    overlap=20,
)
write_facts_jsonl(facts, FACTS_JSONL)
print(f'Extracted {len(facts)} facts')
print(f'Wrote review file: {FACTS_JSONL}')


[graph-ingest] starting fact extraction with model='qwen3.5:9b', ollama_url='http://localhost:11434', max_chars=500, overlap=20
[graph-ingest] document 1: title='Zhuk (radar)', source_id=02bea5aaf367bd98, type=wikipedia, text_chars=19065, chunks=39
[graph-ingest] document 1 chunk 1/39: sending 500 chars to Ollama


{"model":"qwen3.5:9b","created_at":"2026-06-25T14:09:54.8622065Z","response":"","thinking":"{   \"facts\": [] }","done":true,"done_reason":"stop","context":[248045,846,198,2523,8385,264,6979,11,26832,716,314,6337,70534,12688,364,264,4437,1928,5721,4618,883,32338,90241,287,321,8595,1506,13,271,5202,2468,4954,25,198,12,4465,4307,1965,1902,381,6681,799,4566,1576,321,4161,745,13,198,12,4980,279,1965,440,313,321,809,424,440,16414,198,12,3054,524,2830,70703,66814,11,58655,11,5853,11,39492,11,8516,8404,7324,2370,11,466,31626,1414,13,198,12,3054,524,14623,279,4566,1576,303,264,886,13,198,12,5272,1923,16687,364,1396,4566,1328,321,886,869,13,198,12,44123,22151,1923,16687,321,2444,5587,4613,4566,8796,13,198,12,3054,524,958,26849,73982,11,31118,11,38566,11,5438,11,819,12688,11,466,12654,10970,72710,13,198,12,1368,279,11540,11222,874,12688,11,460,6681,5046,65756,763,1249,7563,198,12,17673,11,460,1132,411,10485,25,198,4754,65756,64331,11170,3147,2901,803,2129,59701,3147,47407,3381,2089,88540,1063,70

[graph-ingest] document 1 chunk 1/39: received 0 chars; preview=''
[graph-ingest] document 1 chunk 1/39: model response was empty
[graph-ingest] document 1 chunk 1/39: normalized 0 fact(s); skipped 0 invalid item(s)
[graph-ingest] document 1 chunk 1/39: added 0 fact(s)
[graph-ingest] document 1 chunk 2/39: sending 500 chars to Ollama


{"model":"qwen3.5:9b","created_at":"2026-06-25T14:10:03.3699496Z","response":"","thinking":"{\n\n\"facts\": [\n    { \"subject\": \"Zhuk-MS\", \"predicate\": \"HAS_EXPORT_DESIGNATION\", \"object\": \"Zhuk-MSE\", \"evidence\": \"Export Designation Zhuk-MSE noted in section 2.7 header.\", \"confidence\": 0.9 }, \n    { \"subject\": \"Sokol PESA development program\", \"predicate\": \"HAS_SUBSYSTEM_VARIANT\", \"object\": \"Zhuk-MF\", \"evidence\": \"Formerly part of the Sokol PESA development per section 2.10 header.\", \"confidence\": 0.9 },\n    { \"subject\": \"N031 series radars\", \"predicate\": \"IS_VARIANT_OF\", \"object\": \"Zhuk-MSF\", \"evidence\": \"Export Designation Zhuk-MSF formerly part of N031 Sokol series noted in section 2.11 header.\", \"confidence\": 0.9 },\n    { \"subject\": \"Zhuk (radar)\", \"predicate\": \"HAS_VARIANT\", \"object\": \"F-GA-45\", \"evidence\": \"Section titles list FGA-35(3D) and Zhuk-A as variants or related entities in section 2 headers.\", \"con

[graph-ingest] document 1 chunk 2/39: received 0 chars; preview=''
[graph-ingest] document 1 chunk 2/39: model response was empty
[graph-ingest] document 1 chunk 2/39: normalized 0 fact(s); skipped 0 invalid item(s)
[graph-ingest] document 1 chunk 2/39: added 0 fact(s)
[graph-ingest] document 1 chunk 3/39: sending 500 chars to Ollama


...

{"model":"qwen3.5:9b","created_at":"2026-06-25T14:10:31.6918122Z","response":"","thinking":"{ \"facts\": [ { \"subject\": \"Zhuk (radar)\", \"predicate\": \"HAS_CAPABILITY\", \"object\": \"look-down/shoot-down capability for air targets against a cluttered or water background\", \"evidence\": \"\\\"has a look-down/shoot-down capability for air targets against a cluttered or water background\\\"\", \"confidence\": 1.0 }, { \"subject\": \"Zhuk (radar)\", \"predicate\": \"HAS_MODE\", \"object\": \"search modes\", \"evidence\": \"\\\"a variety of search modes are available\\\"\", \"confidence\": 1.0 }, { \"subject\": \"Zhuk (radar)\", \"predicate\": \"HAS_TRACK_FEATURE\", \"object\": \"track while scan mode\", \"evidence\": \"\\\"engaged at once while searching for new targets in track while scan mode\\\"\", \"confidence\": 1.0 } ] }","done":true,"done_reason":"stop","context":[248045,846,198,2523,8385,264,6979,11,26832,716,314,6337,70534,12688,364,264,4437,1928,5721,4618,883,32338,90241,2

[graph-ingest] document 1 chunk 5/39: received 0 chars; preview=''
[graph-ingest] document 1 chunk 5/39: model response was empty
[graph-ingest] document 1 chunk 5/39: normalized 0 fact(s); skipped 0 invalid item(s)
[graph-ingest] document 1 chunk 5/39: added 0 fact(s)
[graph-ingest] document 1 chunk 6/39: sending 500 chars to Ollama


{"model":"qwen3.5:9b","created_at":"2026-06-25T14:10:42.5815393Z","response":"","thinking":"{ \"facts\": [ { \"subject\": \"Zhuk\", \"predicate\": \"HAS_CAPABILITY\", \"object\": \"lock onto visually acquired target\", \"evidence\": \"locking onto a visually acquired target\", \"confidence\": 1.0 }, { \"subject\": \"Zhuk\", \"predicate\": \"HAS_FEATURE\", \"object\": \"boresight available\", \"evidence\": \"iew, boresight and slewable are available as well\", \"confidence\": 1.0 }, { \"subject\": \"Zhuk\", \"predicate\": \"HAS_FEATURE\", \"object\": \"slewable mode available\", \"evidence\": \"iew, boresight and slewable are available as well\", \"confidence\": 1.0 }, { \"subject\": \"Zhuk radar\", \"predicate\": \"DETECTS\", \"object\": \"helicopters in hover\", \"evidence\": \"track and engage helicopters even when they are hovering\", \"confidence\": 1.0 }, { \"subject\": \"Zhuk\", \"predicate\": \"HAS_ROLE\", \"object\": \"provide targeting for air-to-air missiles\", \"evidence\": 

[graph-ingest] document 1 chunk 6/39: received 0 chars; preview=''
[graph-ingest] document 1 chunk 6/39: model response was empty
[graph-ingest] document 1 chunk 6/39: normalized 0 fact(s); skipped 0 invalid item(s)
[graph-ingest] document 1 chunk 6/39: added 0 fact(s)
[graph-ingest] document 1 chunk 7/39: sending 500 chars to Ollama


KeyboardInterrupt: 

## 5. Populate Neo4j with `graph_ingest.py`

`populate_neo4j` creates the same schema used by the repository CLI:

- `Entity(id, name)`
- `Source(id, source_type, locator)`
- `FACT(predicate, source_id, evidence, confidence)`
- `MENTIONED_IN`

If you want a clean graph, clear the target Neo4j database before running this population cell.

If `NEO4J_PASSWORD` is left as `None` or an empty string, the population cell stops before opening a Neo4j driver and asks you to set the password.


In [ ]:
populate_neo4j(facts, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
print(f'Populated Neo4j database: {NEO4J_URI}')


## 6. Query examples

The queries below inspect the graph generated by `populate_neo4j`. They intentionally work with the generic `Entity`/`FACT` schema from `graph_ingest.py` instead of introducing a separate notebook-only schema.


In [ ]:
import pandas as pd
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
session_kwargs = {'database': NEO4J_DATABASE} if NEO4J_DATABASE else {}
session = driver.session(**session_kwargs)

def show(query: str, params: dict | None = None):
    result = session.run(query, params or {})
    rows = pd.DataFrame([record.data() for record in result])
    display(rows)
    return rows


In [ ]:
show('''
MATCH (source:Source)
RETURN source.source_type AS source_type, source.locator AS locator
ORDER BY locator
''')


In [ ]:
show('''
MATCH (subject:Entity)-[fact:FACT]->(object:Entity)
WHERE lower(subject.name) CONTAINS 'radar'
   OR lower(object.name) CONTAINS 'radar'
   OR lower(fact.predicate) CONTAINS 'sensor'
RETURN subject.name AS subject, fact.predicate AS predicate, object.name AS object,
       fact.confidence AS confidence, fact.evidence AS evidence
ORDER BY confidence DESC
LIMIT 50
''')


In [ ]:
show('''
MATCH (subject:Entity)-[fact:FACT]->(object:Entity)
WHERE lower(subject.name) CONTAINS 'mig-29'
   OR lower(object.name) CONTAINS 'mig-29'
   OR lower(subject.name) CONTAINS 'air force'
   OR lower(object.name) CONTAINS 'air force'
RETURN subject.name AS subject, fact.predicate AS predicate, object.name AS object,
       fact.confidence AS confidence, fact.evidence AS evidence
ORDER BY confidence DESC
LIMIT 50
''')


## 7. Equivalent CLI command

The same module is also exposed by the repository CLI. This notebook form is useful for iterative review, while the command below is better for repeatable batch runs.


In [ ]:
cli = [
    'python -m combat_id_calibration ingest-graph',
    f'  --neo4j-uri {NEO4J_URI}',
    f'  --neo4j-user {NEO4J_USER}',
    '  --neo4j-password "$NEO4J_PASSWORD"',
    f'  --facts-jsonl {FACTS_JSONL}',
    f'  --model {MODEL}',
    f'  --ollama-url {OLLAMA_URL}',
    *[f'  --wikipedia {url}' for url in WIKIPEDIA_URLS],
]
print(' \
'.join(cli))
